## 第1章　数、变量与函数 —— 用Python重温初中数学

> 本章目标：建立"数学概念 → Python 表达"的思维反射。理解 float 的精度陷阱（这不是 Python 的 bug，而是所有计算机的共性），掌握 NumPy 数组为什么比 Python 列表快 10-100 倍，学会用 matplotlib 将任何函数可视化。
> 前置知识：会用 Python 写简单循环、定义变量。



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print("env ready")



### 1.1　整数、浮点数与布尔值 —— 计算机里的"数"不是数学里的"数"

打开任何一个 Python 环境，输入 `0.1 + 0.2`，你会看到 `0.30000000000000004` 而不是 `0.3`。这不是 Python 的 bug——C、Java、JavaScript、Rust 都会给出同样的结果。这是所有计算机共享的"基因缺陷"：**浮点数用二进制近似小数，就像十进制无法精确表示 1/3 = 0.333... 一样，二进制也无法精确表示 0.1。**

这个看似微小的问题在深度学习中会被放大成一个真实的工程挑战：一个 Transformer 模型训练一天要执行数十亿次浮点运算，每次运算的舍入误差如果累积起来，足以让梯度方向偏航。反过来，**理解浮点数的边界，你就能理解为什么训练需要混合精度（第 20 章）、为什么 softmax 在大数值上会溢出（第 22 章）。**

先从基本类型开始。Python 有三种基础数值类型：

- **int（整数）**：精确，无误差。`2 + 3` 永远是 `5`。Python 的 int 没有大小上限（不像 C 的 `int32` 最多到 2³¹−1）。
- **float（浮点数）**：IEEE 754 双精度（64 bits），用二进制近似小数。能表示的范围是 ±1.8×10³⁰⁸，但精度只有约 15-17 位有效数字。
- **bool（布尔值）**：`True` / `False`，本质是 int 的子类——`True == 1`，`False == 0`。这让你可以 `sum([True, False, True])` 得到 `2`。

📐 **定义　IEEE 754 浮点数（IEEE 754 Floating Point）**：用 1 比特符号位 + 11 比特指数 + 52 比特尾数（共 64 bits）近似存储实数。约 15-17 位十进制有效数字，超出部分被舍入。0.1 在二进制中是无限循环小数 `0.0001100110011...`，存为 float 时被截断，导致 0.1 + 0.2 ≠ 0.3。

💻 **代码　亲手验证：类型检测、浮点陷阱、布尔运算**




In [ ]:
# 环境：Python 3.10+，本节仅需内置类型，无需第三方库

# ===== 1. 基本类型 =====
a = 42              # int——精确，内存占用可变（Python 自动处理）
b = 3.14159         # float——64-bit 二进制近似值
c = True            # bool——本质是 int 的 1/0

print(f"a={a}, type={type(a).__name__}")   # int
print(f"b={b}, type={type(b).__name__}")   # float
print(f"c={c}, type={type(c).__name__}")   # bool
print(f"True + True = {True + True}")      # 2 —— bool 本质是 int！

# ===== 2. 浮点精度陷阱——这不是 bug =====
x = 0.1 + 0.2
print(f"\n0.1 + 0.2 = {x}")               # 0.30000000000000004
print(f"0.1 + 0.2 == 0.3 ? {x == 0.3}")   # False！
# 看看到底差多少
print(f"实际差值: {x - 0.3:.2e}")          # ~5.5e-17 —— 在 15 位精度边界上

# ===== 3. 正确的浮点比较：用容差法 =====
EPS = 1e-10  # 容差——只要差异小于这个值，就认为"相等"
print(f"\n容差法判断: {abs(x - 0.3) < EPS}")  # True ✓

# math.isclose 是标准做法
import math
print(f"math.isclose: {math.isclose(0.1 + 0.2, 0.3)}")  # True ✓

# ===== 4. 浮点数的极限 =====
# 小数精度边界
print(f"\n1.0 + 1e-16 == 1.0 ? {1.0 + 1e-16 == 1.0}")   # True——1e-16 在 float 精度之下
print(f"1.0 + 1e-15 == 1.0 ? {1.0 + 1e-15 == 1.0}")    # False——1e-15 刚好能表示

# 整数范围——Python int 没有上限！
big = 2 ** 1000
print(f"\n2^1000 = {big}")  # 一个 302 位的整数，Python 轻松处理
print(f"2^1000 的位数: {len(str(big))}")




> **关键洞察**：`0.1 + 0.2 == 0.3` 返回 `False` 不是因为 Python 算错了，而是因为在二进制世界中，0.1 和 0.2 根本不存在——只有它们的近似值。深度学习中的每一个 forward pass、每一个 loss 计算都在不断累加这种近似误差。Pytorch 提供 `float32`（7 位精度）和 `bfloat16`（7 位精度但范围大）两种低精度格式来加速训练——它们精度更低，误差更大，但靠大量随机采样"平均掉"了误差（第 20 章详解）。

🔗 **AI 连接**：混合精度训练（第 20 章）用 `float16` 做前向/反向传播以加速 2-3 倍，同时用 `float32` 保存权重副本以控制累积误差。Softmax 在输入接近 1000 时会因 `exp(1000)` 超出 `float32` 范围而返回 `nan`——这是第 22 章要解决的核心问题。**理解 float 的本质，就是你理解 AI 训练中所有数值问题的起点。**

---

### 1.2　从列表到 NumPy 数组 —— AI 为什么不用 Python 列表？

你可能会觉得：Python 列表 `[1, 2, 3, 4, 5]` 挺好用的，为什么搞 AI 的人全都在用 NumPy 数组 `np.array([1, 2, 3, 4, 5])`？两者看起来差不多——但底层原理决定了性能差距高达 10-100 倍。

Python 列表是一个**指针数组**：列表里的每个元素其实是一个指向内存中某个 Python 对象的指针。`[1, "hello", 3.14]` 里面存的不是值，而是三个指针，分别指向三个不同类型的对象。这种设计给了 Python 极大的灵活性（一个列表可以混合任何类型），但代价是：

1. **内存不连续**：值散落在内存各处，CPU 缓存命中率低
2. **每次访问都要解引用**：先读指针，再通过指针找到值——两步操作
3. **无法利用 SIMD 指令**：现代 CPU 可以一次操作 4 个 float32，但前提是它们在内存中紧挨着

而 NumPy 数组 `ndarray` 是**同类型、连续内存**的——它像一个 C 数组：所有元素的类型相同（比如全是 `float64`），一个接一个紧密排列在内存中。没有指针跳转，不需要类型检查，CPU 可以把一整块数据一次性加载到缓存里，用向量化指令并行处理。

> 📐 **定义　NumPy 数组（ndarray）**：同类型元素连续存储于内存的多维数组。所有元素共享同一 `dtype`（如 `float64`）。运算在 C 层面执行，Python 只做调度——这就是"向量化"比 Python 循环快的根本原因。

💻 **代码　列表 vs 数组的全方位对比：内存、速度、便利性**




In [ ]:
import numpy as np
import time

# ===== 1. 内存占用对比 =====
py_list = list(range(1_000_000))           # Python 列表：100 万个 Python int 对象
np_arr = np.arange(1_000_000, dtype=np.int64)  # NumPy 数组：100 万个 int64 连续排列

# 每个 Python int 对象约 28 字节（Python 3.10+），加上 8 字节指针 ≈ 36 MB
# np.int64 数组每个元素 8 字节，总约 8 MB——差了 4-5 倍
print(f"Python 列表内存 ≈ {py_list.__sizeof__() / 1e6:.1f} MB (仅指针数组)")
print(f"NumPy 数组内存 = {np_arr.nbytes / 1e6:.1f} MB (实际数据)")

# ===== 2. 运算速度对比：逐个元素平方 =====
# Python 循环方式
t0 = time.perf_counter()
squared_list = [x ** 2 for x in py_list]
t1 = time.perf_counter()

# NumPy 向量化方式
t2 = time.perf_counter()
squared_arr = np_arr ** 2   # 一行！在 C 层面并行执行
t3 = time.perf_counter()

print(f"\nPython 列表推导式: {t1 - t0:.4f}s")
print(f"NumPy 向量化运算:   {t3 - t2:.4f}s")
print(f"NumPy 快了 {(t1-t0)/(t3-t2):.1f} 倍 ✓")

# ===== 3. NumPy 的核心优势：操作即操作，不用写循环 =====
arr = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(f"\n原数组: {arr}")
print(f"全部 + 10: {arr + 10}")              # 广播到每个元素
print(f"全部平方: {arr ** 2}")
print(f"大于 3 的元素: {arr[arr > 3]}")       # 布尔索引——不用写 if
print(f"总和: {arr.sum()}, 均值: {arr.mean():.2f}, 标准差: {arr.std():.2f}")
# 这些操作在深度学习中无处不在——下一行本质上就是 X @ W + b




> **关键洞察**：Python 列表是为"灵活性"设计的，NumPy 数组是为"批量计算"设计的。AI 训练的核心运算是矩阵乘法——`(32, 784) @ (784, 256)`——这意味着要对 32×784×256 = 640 万个浮点数逐对相乘再求和。如果用 Python 循环，一次 epoch 就要数分钟；用 NumPy 的 C 底层 + SIMD 并行，几毫秒完成。本书后面会看到，PyTorch 的张量继承了 NumPy 数组的所有优点，并且可以搬上 GPU 进一步提速 100 倍。

🔗 **AI 连接**：PyTorch 的 `torch.Tensor` 和 NumPy 的 `ndarray` 共享同一个设计哲学——连续内存、同类型、向量化运算。从第 4 章开始的每一行 `torch.randn(32, 784) @ torch.randn(784, 256)` 都建立在你此刻理解的"连续内存"之上。如果你带着 "NumPy = GPU 前的 Python 张量" 这个认知继续读，后续每一章的性能讨论都无需额外解释。

---

### 1.3　函数就是"输入→输出"的盒子 —— 数学与代码的桥梁

数学课本上写 f(x) = x²，Python 里写 `def f(x): return x**2`。表面上看是两种写法，本质上是同一个东西：**函数 = 输入到输出的映射规则**。这个等式如此重要，以至于整本书都可以看作它的展开：

$$神经网络 = f(x; θ) = 一个将输入图片映射到"猫/狗/..."概率的巨大函数$$

在深入这个巨大函数之前，你需要一个"可视化函数"的通用工具——给定任意函数定义、任意输入范围，画出它的图像。这个工具将贯穿全书：画激活函数、画损失曲面、画梯度下降轨迹、画 softmax 曲线——全都基于同一个模板。

📐 **定义　函数（Function）**：对于每个输入 x，有唯一确定的输出 f(x)。Python 中用 `def` 定义。`np.linspace(start, end, n)` 在区间内均匀采样 n 个点，是画连续函数图像的标配工具。

💻 **代码　matplotlib 最简绘图模板——两行画线，五行成图**




In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ===== 最简模板：只需学会这个，全书绘图都能复用 =====
def plot_function(f, x_range=(-3, 3), n_points=200,
                  title="函数图像", xlabel="x", ylabel="f(x)"):
    """
    通用绘图模板——画任意单变量函数的图像。
    f: 一个 Python 函数，如 lambda x: x**2
    """
    x = np.linspace(x_range[0], x_range[1], n_points)  # 在 x_range 内均匀采样
    y = f(x)                                            # 向量化求值——一次算完 200 个点
    plt.figure(figsize=(7, 4))
    plt.plot(x, y, 'b-', linewidth=2)
    plt.axhline(y=0, color='gray', linewidth=0.5)       # x 轴参考线
    plt.axvline(x=0, color='gray', linewidth=0.5)       # y 轴参考线
    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title); plt.grid(alpha=0.3); plt.show()

# ===== 画直线 y = 2x + 3 =====
def f_linear(x):
    return 2 * x + 3

plot_function(f_linear, x_range=(-4, 2),
              title="$f(x) = 2x + 3$（线性函数：直线，斜率恒定）")

# ===== 画抛物线 y = x² =====
def f_quadratic(x):
    return x ** 2

plot_function(f_quadratic, x_range=(-3, 3),
              title="$f(x) = x^2$（二次函数：曲线，斜率在变）")

# ===== 两个函数放一起对比 =====
x = np.linspace(-3, 3, 200)
plt.figure(figsize=(7, 4))
plt.plot(x, 2*x + 3, 'b-', linewidth=2, label="$y=2x+3$（线性）")
plt.plot(x, x**2, 'r-', linewidth=2, label="$y=x^2$（二次）")
plt.axhline(y=0, color='gray', linewidth=0.5)
plt.axvline(x=0, color='gray', linewidth=0.5)
plt.xlabel("x"); plt.ylabel("y"); plt.title("直线 vs 抛物线 —— 常量斜率 vs 变化斜率")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("关键观察：直线的倾斜度处处相同，抛物线的倾斜度随 x 而变——这就是'导数'的核心直觉。")




> **关键洞察**：`x = np.linspace(-3, 3, 200)` 这行代码是全书最重要的工具模式——把连续区间变成 200 个离散采样点，对每个点求 f(x)，然后连接成线。深度学习中几乎所有"曲线"都是这样画出来的：损失曲线、学习率衰减曲线、激活函数曲线、概率分布曲线。你刚刚学会的不是画图，而是**将数学变成直觉**的通用方法论。

🔗 **AI 连接**：`plot_function` 这个模板将在第 2 章画割线趋近切线的动画、第 10 章画各激活函数的导函数、第 12 章画损失曲面上的梯度下降轨迹、第 22 章画不同温度的 softmax 概率分布曲线、第 24 章画 SGD/Momentum/Adam 的收敛路径。掌握它，你就有了可视化全书任何一个数学概念的"万能钥匙"。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）`0.1 + 0.2 == 0.3` 为什么返回 `False`？这不是 Python 的 bug，那根本原因是什么？用一句话回答。
2. （概念）如果你要存储 100 万个 float 数，Python 列表和 NumPy 数组各占大约多少内存？为什么 NumPy 数组的运算速度远快于列表？
3. （概念）`np.linspace(-3, 3, 200)` 做了什么？为什么画函数图像需要它？如果 `n_points=5` 会怎样？
4. （代码）在 Python 中输入 `0.1 + 0.1 + 0.1 - 0.3`，结果是多少？解释它为什么不是 0。然后用 `math.isclose` 验证它"实质上等于 0"。
5. （代码）用 `plot_function` 模板画出以下三个函数在 x∈[−2, 2] 上的图像，放在同一张图上对比，添加图例标注：
   - `f₁(x) = x³`
   - `f₂(x) = 1 / (1 + np.exp(-x))` （这个函数叫 sigmoid——第 27 章的主角）
   - `f₃(x) = np.maximum(0, x)` （这个函数叫 ReLU——神经网络最常用的激活函数）
---
> 预览：下一章——**变化的直觉**，用割线逼近切线，从"平均变化"走向"瞬时变化"。
